# Detector Comparison — Adaptive Road Vision (YOLO11n vs YOLO11s)

Compare only the two detectors used in this project:
1. Detector 1 — YOLO11n (nano)
2. Detector 2 — YOLO11s (small)

This notebook does **not retrain models**. It loads the saved final test metrics and per-class metrics from the two training notebooks, displays a side-by-side comparison, and saves comparison artifacts. Run the Detector 1 and Detector 2 training notebooks first.


In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Project paths requested by the user.
PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
RESULT_ROOT = PROJECT_ROOT / "results" / "detection"
MODEL_ROOT = PROJECT_ROOT / "models" / "detection"

# Create output folders if they do not exist; do not fail before reporting missing inputs.
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
else:
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. Check that the folder exists on this computer."
    )

print("Project root:", PROJECT_ROOT)
print("Results:", RESULT_ROOT)
print("Models:", MODEL_ROOT)


## 1. Expected artifacts

The comparison reads the final reports and per-class test metric CSVs generated by the Detector 1 (YOLO11n) and Detector 2 (YOLO11s) training notebooks. Both models should use the same dataset split, class names/order, and evaluation settings for a fair comparison.


In [ ]:
detectors = [
    {
        "detector": "Detector 1",
        "model": "YOLO11n",
        "report": RESULT_ROOT / "detector_1_final_report.json",
        "class_csv": RESULT_ROOT / "detector_1_test_class_metrics.csv",
        "checkpoint": MODEL_ROOT / "detector_1_best.pt",
    },
    {
        "detector": "Detector 2",
        "model": "YOLO11s",
        "report": RESULT_ROOT / "detector_2_final_report.json",
        "class_csv": RESULT_ROOT / "detector_2_test_class_metrics.csv",
        "checkpoint": MODEL_ROOT / "detector_2_best.pt",
    },
]

status_rows = []
for d in detectors:
    status_rows.append({
        "detector": d["detector"],
        "model": d["model"],
        "report_exists": d["report"].is_file(),
        "class_metrics_exists": d["class_csv"].is_file(),
        "checkpoint_exists": d["checkpoint"].is_file(),
    })

status_df = pd.DataFrame(status_rows)
display(status_df)

missing_reports = [d["report"].name for d in detectors if not d["report"].is_file()]
if missing_reports:
    print("WARNING: Missing final report(s):", ", ".join(missing_reports))
    print("Run the corresponding detector training notebook(s) before interpreting the comparison.")
else:
    print("Both detector final reports are present.")


## 2. Load test metrics

Only metrics saved by the training notebooks are loaded. Missing values are left as `NaN`; this notebook does not invent or recompute them.


In [ ]:
summary_rows = []
reports = {}

def first_present(mapping, *keys):
    """Return the first non-None metric from supported report key names."""
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        value = mapping.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                continue
    return None

for d in detectors:
    if not d["report"].is_file():
        continue

    with open(d["report"], "r", encoding="utf-8") as f:
        report = json.load(f)
    reports[d["detector"]] = report

    # Detector 1's training notebook saves test metrics under test_report;
    # Detector 2 may save them under test_metrics. Support both.
    test = report.get("test_metrics") or report.get("test_report") or {}
    if not isinstance(test, dict):
        test = {}

    summary_rows.append({
        "detector": d["detector"],
        "model": d["model"],
        "precision": first_present(test, "precision", "P", "metrics/precision(B)"),
        "recall": first_present(test, "recall", "R", "metrics/recall(B)"),
        "mAP50": first_present(test, "map50", "mAP50", "mAP@0.50", "metrics/mAP50(B)"),
        "mAP50-95": first_present(test, "map50_95", "mAP50-95", "map", "mAP@0.50:0.95", "metrics/mAP50-95(B)"),
        "checkpoint_exists": d["checkpoint"].is_file(),
    })

comparison_df = pd.DataFrame(summary_rows, columns=[
    "detector", "model", "precision", "recall", "mAP50", "mAP50-95", "checkpoint_exists"
])
if comparison_df.empty:
    missing = [str(d["report"]) for d in detectors if not d["report"].is_file()]
    raise FileNotFoundError(
        "No detector final reports were found. Expected reports at:\n" + "\n".join(missing) +
        "\nRun Detector 1 (YOLO11n) and Detector 2 (YOLO11s) training notebooks first."
    )

for col in ["precision", "recall", "mAP50", "mAP50-95"]:
    comparison_df[col] = pd.to_numeric(comparison_df[col], errors="coerce")

display(comparison_df)


## 3. Per-class test comparison

This section combines saved per-class test metrics for YOLO11n and YOLO11s. Missing CSVs are reported rather than fabricated.


In [ ]:
per_class_frames = []
for d in detectors:
    if not d["class_csv"].is_file():
        print(f"Per-class metrics not found for {d['model']}: {d['class_csv']}")
        continue
    df = pd.read_csv(d["class_csv"])
    df.insert(0, "detector", d["detector"])
    df.insert(1, "model", d["model"])
    per_class_frames.append(df)

if per_class_frames:
    per_class_df = pd.concat(per_class_frames, ignore_index=True)
    display(per_class_df)
else:
    per_class_df = pd.DataFrame()
    print("No per-class metric CSVs were found; the overall comparison can still be reviewed if reports exist.")


In [ ]:
# Pivot mAP50-95 by class for a direct two-model comparison.
if not per_class_df.empty and {"class_name", "map50_95"}.issubset(per_class_df.columns):
    class_map_table = per_class_df.pivot_table(
        index="class_name", columns="model", values="map50_95", aggfunc="first"
    )
    display(class_map_table)
else:
    class_map_table = pd.DataFrame()
    print("A class map table is unavailable because the CSVs lack class_name/map50_95 columns.")


## 4. Metric visualization

These plots show measured test metrics for the two models. Interpret them alongside inference speed, model size, and per-class performance; higher accuracy alone does not establish the best deployment choice.


In [ ]:
plot_df = comparison_df.set_index("model")
metrics_to_plot = [
    ("mAP50", "Test mAP@0.50"),
    ("mAP50-95", "Test mAP@0.50:0.95"),
    ("precision", "Test Precision"),
    ("recall", "Test Recall"),
]

for metric, title in metrics_to_plot:
    if metric not in plot_df.columns or plot_df[metric].notna().sum() == 0:
        print(f"Skipping {title}: no numeric values were saved.")
        continue
    ax = plot_df[metric].plot(kind="bar", figsize=(7, 4), rot=0)
    ax.set_ylabel(metric)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


## 5. Compact comparison table

This table summarizes the saved test metrics. Model parameter counts are not inferred from metrics; use the YOLO model specifications or measured checkpoint sizes separately if needed.


In [ ]:
model_size_table = comparison_df[[
    "detector", "model", "precision", "recall", "mAP50", "mAP50-95", "checkpoint_exists"
]].copy()
display(model_size_table)


## 6. Save comparison artifacts

The CSV and JSON outputs are written to `results/detection/` and can be used by the final detector-selection notebook. Only the two selected models are included.


In [ ]:
# Explicitly display the YOLO11n and YOLO11s summary before saving.
display(comparison_df.style.format({
    "precision": "{:.4f}", "recall": "{:.4f}",
    "mAP50": "{:.4f}", "mAP50-95": "{:.4f}"
}, na_rep="N/A"))

# Print the YOLO11n artifacts so missing outputs are easy to diagnose.
yolo11n = detectors[0]
print("YOLO11n report:", yolo11n["report"], "exists =", yolo11n["report"].is_file())
print("YOLO11n checkpoint:", yolo11n["checkpoint"], "exists =", yolo11n["checkpoint"].is_file())
print("YOLO11n per-class CSV:", yolo11n["class_csv"], "exists =", yolo11n["class_csv"].is_file())


In [ ]:
comparison_csv = RESULT_ROOT / "detector_comparison_summary.csv"
comparison_df.to_csv(comparison_csv, index=False)

status_csv = RESULT_ROOT / "detector_comparison_artifact_status.csv"
status_df.to_csv(status_csv, index=False)

if not per_class_df.empty:
    per_class_csv = RESULT_ROOT / "detector_comparison_per_class.csv"
    per_class_df.to_csv(per_class_csv, index=False)
else:
    per_class_csv = None

if not class_map_table.empty:
    class_map_csv = RESULT_ROOT / "detector_comparison_class_map50_95.csv"
    class_map_table.to_csv(class_map_csv)
else:
    class_map_csv = None

comparison_report = {
    "project_root": str(PROJECT_ROOT),
    "detectors": [{"detector": d["detector"], "model": d["model"]} for d in detectors],
    "source_reports": {d["detector"]: str(d["report"]) for d in detectors},
    "comparison_summary_csv": str(comparison_csv),
    "status_csv": str(status_csv),
    "per_class_csv": str(per_class_csv) if per_class_csv else None,
    "class_map50_95_csv": str(class_map_csv) if class_map_csv else None,
    "note": (
        "Descriptive test-set metrics loaded from the YOLO11n and YOLO11s training reports. "
        "Use identical test data and class order for a fair comparison. This notebook does not retrain models."
    ),
}

report_json = RESULT_ROOT / "detector_comparison_report.json"
report_json.write_text(json.dumps(comparison_report, indent=2), encoding="utf-8")

print("Saved:", comparison_csv)
print("Saved:", status_csv)
print("Saved:", report_json)
if per_class_csv:
    print("Saved:", per_class_csv)
if class_map_csv:
    print("Saved:", class_map_csv)


## Next step

Use the final detector-selection notebook to select between **YOLO11n** and **YOLO11s** using the measured test metrics together with inference latency, model size, and per-class performance. Do not select a model based on the held-out test set repeatedly during training; use validation metrics for tuning and reserve the test set for final reporting.
